# SVCM — Terminology Consumer — Cas de test nominal

**Profil** : IHE ITI SVCM  
**Acteur testé** : SVCM-Terminology Consumer (`PAT_DEVICE_COLLECTEUR_ANALYSEUR_DE_DONNEES_CAD_CDM`)  
**Répertoire cible** : SMT national ANS (`smt.esante.gouv.fr`)  
**Référence test** : [SVCM_Cas de test nominal](https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12231&cid=29222)  
**Référence FHIR server** : [Global features FHIR Server v5](https://esante.gouv.fr/sites/default/files/media/document/Global_features_FHIR_Server_version_finale_v5.pdf)

Ce notebook exécute l'intégralité des étapes du cas de test nominal et constitue la **trace + preuve** à soumettre au moniteur Gazelle.

## Configuration

In [1]:
import requests
import json
import time
import logging
from datetime import datetime
from urllib.parse import quote
from IPython.display import display, JSON, Markdown

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_BASE  = "https://smt.esante.gouv.fr/api"
WP_BASE   = "https://smt.esante.gouv.fr/wp-json/ans"
DOMAIN    = "smt.esante.gouv.fr"

# Remplacer par la clé API complète
API_KEY = "ZiaoDxF8Je0CfBNzu..."

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM"
}
HEADERS_API = {
    "accept": "*/*",
    "X-API-KEY": API_KEY,
    "User-Agent": "SVCM-Consumer-CAD-CDM"
}

# ── helpers ──────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_json(filename, data):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ Sauvegardé : {filename}")

def print_keys(data, *keys):
    """Affiche un sous-ensemble de champs d'un dict pour la preuve."""
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

def extract_concepts_recursive(concepts, prefix=""):
    flat = {}
    for c in concepts:
        code = c.get("code")
        display_val = c.get("display")
        flat[code] = {"display": display_val, "path": prefix + "/" + code}
        if "concept" in c:
            flat.update(extract_concepts_recursive(c["concept"], prefix + "/" + code))
    return flat

print("Configuration OK — prêt à exécuter les étapes.")

Configuration OK — prêt à exécuter les étapes.


---
## Étapes 10–30 — Récupération du CapabilityStatement général

**Transaction** : SVCM-ITI-95  
**Requête** : `GET /fhir/metadata?_format=json`  
**Objectif** : Vérifier que le SMT expose bien un CapabilityStatement FHIR valide et l'intégrer.

In [2]:
# Étape 10 — Construction + envoi de la requête
# Étape 20 — Réception et intégration
# Étape 30 — PREUVE

url_cs = f"{FHIR_BASE}/metadata?_format=json"
r_cs = http_get(url_cs)
cs = r_cs.json()

save_json("smt-cs.json", cs)

print("\n[PREUVE étape 30] Champs clés du CapabilityStatement :")
print_keys(cs, "resourceType", "status", "date", "kind", "fhirVersion", "software")

print(f"\nNombre de ressources supportées : {len(cs.get('rest', [{}])[0].get('resource', []))}")
resource_types = [r['type'] for r in cs.get('rest', [{}])[0].get('resource', [])]
print("Types :", resource_types)

  → GET https://smt.esante.gouv.fr/fhir/metadata?_format=json
  ✓ Sauvegardé : smt-cs.json

[PREUVE étape 30] Champs clés du CapabilityStatement :
{
  "resourceType": "CapabilityStatement",
  "status": "active",
  "date": "2026-03-04T07:03:35.304+01:00",
  "kind": "instance",
  "fhirVersion": "4.0.1",
  "software": {
    "extension": [
      {
        "url": "http://ontoserver.csiro.au/profiles/ontoserver-versions",
        "extension": [
          {
            "url": "snomedIndexVersion",
            "valueString": "2.0.14"
          },
          {
            "url": "loincIndexVersion",
            "valueString": "2.0.6"
          },
          {
            "url": "fhirIndexVersion",
            "valueString": "2.8.0"
          }
        ]
      }
    ],
    "name": "Ontoserver®",
    "version": "6.22.6",
    "releaseDate": "2025-05-14T06:51:16+02:00"
  }
}

Nombre de ressources supportées : 146
Types : ['Account', 'ActivityDefinition', 'AdverseEvent', 'AllergyIntolerance', 'Appoint

---
## Étapes 50–70 — Récupération du CapabilityStatement terminologique

**Transaction** : SVCM-ITI-95  
**Requête** : `GET /fhir/metadata?_format=json&mode=terminology`  
**Objectif** : Vérifier les capacités spécifiquement terminologiques du SMT.

In [3]:
# Étape 50 — Construction + envoi
# Étape 60 — Réception et intégration
# Étape 70 — PREUVE

url_tc = f"{FHIR_BASE}/metadata?_format=json&mode=terminology"
r_tc = http_get(url_tc)
tc = r_tc.json()

save_json("smt-tc.json", tc)

print("\n[PREUVE étape 70] Champs clés du CapabilityStatement terminologique :")
print_keys(tc, "resourceType", "status", "date", "kind", "fhirVersion")

# Opérations terminologiques exposées
operations = [
    op.get("name")
    for r in tc.get('rest', [{}])[0].get('resource', [])
    for op in r.get('operation', [])
]
if operations:
    print("\nOpérations terminologiques disponibles :", sorted(set(operations)))

  → GET https://smt.esante.gouv.fr/fhir/metadata?_format=json&mode=terminology
  ✓ Sauvegardé : smt-tc.json

[PREUVE étape 70] Champs clés du CapabilityStatement terminologique :
{
  "resourceType": "TerminologyCapabilities",
  "status": "active",
  "date": "2026-03-04T12:02:43+01:00",
  "kind": "capability"
}


---
## Étapes 90–110 — Récupération d'une terminologie : TRE-R256-TypeMessagerie

**Transaction** : SVCM-ITI-95  
**Requête** : `GET /fhir/CodeSystem?url=https://mos.esante.gouv.fr/NOS/TRE_R256-TypeMessagerie/FHIR/TRE-R256-TypeMessagerie&_format=json`  
**Objectif** : Retrouver la terminologie par URL canonique, vérifier `content=complete`, récupérer son `id` et `versionId`.

In [4]:
# Étape 90 — Construction + envoi
TRE256_URL = "https://mos.esante.gouv.fr/NOS/TRE_R256-TypeMessagerie/FHIR/TRE-R256-TypeMessagerie"
url_tre256 = f"{FHIR_BASE}/CodeSystem?url={quote(TRE256_URL)}&_format=json"

r_tre256 = http_get(url_tre256)
bundle_tre256 = r_tre256.json()

save_json("smt-tre256.json", bundle_tre256)

# Étape 100 — Intégration : parsing de resource.entry.content="complete"
entries = bundle_tre256.get("entry", [])
print(f"Entrées trouvées : {len(entries)}")

cs_entry = entries[0]
cs_resource = cs_entry["resource"]
full_url = cs_entry.get("fullUrl", "")

print(f"  content = {cs_resource.get('content')}  (attendu: complete)")
assert cs_resource.get("content") == "complete", "ERREUR : content != complete"

# Étape 110 — PREUVE : GET par fullUrl, parsing de id et versionId
print(f"\n[PREUVE étape 110] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
cs_direct = r_direct.json()

cs_id      = cs_direct.get("id")
version_id = cs_direct.get("meta", {}).get("versionId")
version    = cs_direct.get("version")

print(f"  id        = {cs_id}")
print(f"  versionId = {version_id}")
print(f"  version   = {version}")
print(f"  name      = {cs_direct.get('name')}")
print(f"  title     = {cs_direct.get('title')}")
print(f"  concepts  = {len(cs_direct.get('concept', []))}")

  → GET https://smt.esante.gouv.fr/fhir/CodeSystem?url=https%3A//mos.esante.gouv.fr/NOS/TRE_R256-TypeMessagerie/FHIR/TRE-R256-TypeMessagerie&_format=json
  ✓ Sauvegardé : smt-tre256.json
Entrées trouvées : 1
  content = complete  (attendu: complete)

[PREUVE étape 110] GET par fullUrl : https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie
  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie?_format=json
  id        = TRE-R256-TypeMessagerie
  versionId = 5
  version   = 20231215120000
  name      = TRE_R256_TypeMessagerie
  title     = None
  concepts  = 4


---
## Étapes 130–150 — Historique et diff entre versions de TRE-R256

**Requêtes** :  
- `GET /fhir/CodeSystem/{id}/_history?_format=json` → compter les versions (`total`)  
- `GET /fhir/CodeSystem/{id}/_history/{total}?_format=json` → version courante  
- `GET /fhir/CodeSystem/{id}/_history/{total-1}?_format=json` → version précédente  

**Objectif** : Calculer les différences (codes ajoutés / supprimés / libellés modifiés) entre les deux dernières versions.

In [5]:
# Étape 130 — Récupération de l'historique
url_hist = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history?_format=json"
r_hist = http_get(url_hist)
hist = r_hist.json()

total_versions = hist.get("total", 0)
print(f"[PREUVE étape 130] Nombre total de versions : {total_versions}")

url_latest = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history/{total_versions}?_format=json"
url_prev   = f"{FHIR_BASE}/CodeSystem/{cs_id}/_history/{total_versions - 1}?_format=json"

print(f"Version courante  : {url_latest}")
print(f"Version précédente: {url_prev}")

cs_latest = http_get(url_latest).json()

# Étape 140 — Version précédente
cs_prev = http_get(url_prev).json()

print(f"\nVersion courante  : {cs_latest.get('version')} ({len(cs_latest.get('concept', []))} concepts)")
print(f"Version précédente: {cs_prev.get('version')} ({len(cs_prev.get('concept', []))} concepts)")

  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie/_history?_format=json
[PREUVE étape 130] Nombre total de versions : 5
Version courante  : https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie/_history/5?_format=json
Version précédente: https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie/_history/4?_format=json
  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie/_history/5?_format=json
  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R256-TypeMessagerie/_history/4?_format=json

Version courante  : 20231215120000 (4 concepts)
Version précédente: 20231215120000 (4 concepts)


In [6]:
# Étape 150 — PREUVE : diff entre les deux versions

c_prev   = extract_concepts_recursive(cs_prev.get("concept", []))
c_latest = extract_concepts_recursive(cs_latest.get("concept", []))

set_prev   = set(c_prev.keys())
set_latest = set(c_latest.keys())

added   = set_latest - set_prev
removed = set_prev - set_latest
changed = {k for k in set_prev & set_latest if c_prev[k]["display"] != c_latest[k]["display"]}

print(f"[PREUVE étape 150] Diff {cs_prev.get('version')} → {cs_latest.get('version')}")
print(f"  Codes ajoutés   : {len(added)}")
print(f"  Codes supprimés : {len(removed)}")
print(f"  Libellés modifiés: {len(changed)}")

if added:
    print("\n  AJOUTÉS :")
    for k in sorted(added):
        print(f"    {k} — {c_latest[k]['display']}")

if removed:
    print("\n  SUPPRIMÉS :")
    for k in sorted(removed):
        print(f"    {k} — {c_prev[k]['display']}")

if changed:
    print("\n  LIBELLÉS MODIFIÉS :")
    for k in sorted(changed):
        print(f"    {k}  AVANT: {c_prev[k]['display']}  APRÈS: {c_latest[k]['display']}")

if not added and not removed and not changed:
    print("  Aucune différence entre les deux versions.")

[PREUVE étape 150] Diff 20231215120000 → 20231215120000
  Codes ajoutés   : 0
  Codes supprimés : 0
  Libellés modifiés: 0
  Aucune différence entre les deux versions.


---
## Étapes 170–190 — Récupération de SNOMED CT International

**Requêtes** :  
- `GET /fhir/CodeSystem?_format=json&url=http://snomed.info/sct` → métadonnées FHIR  
- `GET /fhir/CodeSystem/900000000000207008-20260301` → fiche détaillée  
- `GET /api/terminologies/list` → liste via API propriétaire  
- `GET /wp-json/ans/terminologies/versions-details?terminologyId=terminologie-snomed-ct` → versions  
- Téléchargement du zip → `snomed-ct.zip`

**Note** : `content = "not-present"` est attendu (SNOMED n'est pas hébergé inline dans le FHIR server).

In [7]:
# Étape 170 — Métadonnées FHIR SNOMED CT International
url_snomed = f"{FHIR_BASE}/CodeSystem?_format=json&url=http://snomed.info/sct"
r_snomed = http_get(url_snomed)
bundle_snomed = r_snomed.json()

print("[PREUVE étape 170] Entrées SNOMED CT international :")
for e in bundle_snomed.get("entry", []):
    res = e.get("resource", {})
    print(f"  fullUrl : {e.get('fullUrl')}")
    print(f"  name    : {res.get('name')}")
    print(f"  version : {res.get('version')}")
    print(f"  title   : {res.get('title')}")
    print(f"  content : {res.get('content')}  (attendu: not-present)")
    print()

  → GET https://smt.esante.gouv.fr/fhir/CodeSystem?_format=json&url=http://snomed.info/sct
[PREUVE étape 170] Entrées SNOMED CT international :
  fullUrl : https://smt.esante.gouv.fr/fhir/CodeSystem/11000315107-20250621
  name    : SNOMED_CT
  version : http://snomed.info/sct/11000315107/version/20250621
  title   : French module
  content : not-present  (attendu: not-present)

  fullUrl : https://smt.esante.gouv.fr/fhir/CodeSystem/900000000000207008-20260301
  name    : SNOMED_CT
  version : http://snomed.info/sct/900000000000207008/version/20260301
  title   : SNOMED CT core
  content : not-present  (attendu: not-present)



In [9]:
# Étape 180 — Fiche détaillée + liste via API propriétaire
url_snomed_detail = f"{FHIR_BASE}/CodeSystem/900000000000207008-20260301"
r_detail = http_get(f"{url_snomed_detail}?_format=json")
snomed_detail = r_detail.json()
print("Détail CodeSystem SNOMED :")
print_keys(snomed_detail, "id", "version", "name", "title", "status", "content")

print("\nListe des terminologies (API) :")
r_list = http_get(f"{API_BASE}/terminologies/list", HEADERS_API)
terminologies = r_list.json()
snomed_entries = [t for t in (terminologies if isinstance(terminologies, list) else [])
                  if "snomed" in str(t).lower()]
print(f"  Entrées SNOMED dans la liste : {len(snomed_entries)}")
for t in snomed_entries:
    print(f"  {t}")

print("\nVersions détaillées SNOMED CT :")
url_versions = f"{WP_BASE}/terminologies/versions-details?terminologyId=terminologie-snomed-ct"
r_versions = http_get(url_versions, HEADERS_API)
versions_data = r_versions.json()
print(json.dumps(versions_data, indent=2, ensure_ascii=False)[:1000])

  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/900000000000207008-20260301?_format=json
Détail CodeSystem SNOMED :
{
  "id": "900000000000207008-20260301",
  "version": "http://snomed.info/sct/900000000000207008/version/20260301",
  "name": "SNOMED_CT",
  "title": "SNOMED CT core",
  "status": "active",
  "content": "not-present"
}

Liste des terminologies (API) :
  → GET https://smt.esante.gouv.fr/api/terminologies/list
  ✗ tentative 1/3 : HTTP 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx/1.18.0 (Ubuntu)</center>
</body>
</html>

  → GET https://smt.esante.gouv.fr/api/terminologies/list
  ✗ tentative 2/3 : HTTP 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx/1.18.0 (Ubuntu)</center>
</body>
</html>

  → GET https://smt.esante.gouv.fr/api/terminologies/list
  ✗ tentative 3/3 : HTTP 401: <

Exception: HTTP 401: <html>
<head><title>401 Authorization Required</title></head>
<body>
<center><h1>401 Authorization Required</h1></center>
<hr><center>nginx/1.18.0 (Ubuntu)</center>
</body>
</html>


In [ ]:
# Étape 190 — PREUVE : téléchargement SNOMED CT zip
snomed_zip_url = (
    f"{WP_BASE}/terminologies/zip"
    "?terminologyId=terminologie-snomed-ct"
    "&version=Janvier%202026%20v1.0"
    "&licenceConsent=true"
    "&dataTransferConsent=true"
)
print(f"Téléchargement : {snomed_zip_url}")
r_zip = http_get(snomed_zip_url, HEADERS_API)
with open("snomed-ct.zip", "wb") as f:
    f.write(r_zip.content)
print(f"✓ snomed-ct.zip — {len(r_zip.content) / 1024 / 1024:.1f} MB")

---
## Étapes 210–230 — Récupération de SNOMED CT France (Extension FR)

**Requêtes** :  
- `GET /fhir/CodeSystem/11000315107-20250621` → fiche détaillée extension FR  
- `GET /wp-json/ans/terminologies/versions-details?terminologyId=terminologie-snomed-ct-fr`  
- Téléchargement du zip → `snomed-ct-fr.zip`

In [ ]:
# Étape 210 — Fiche détaillée SNOMED CT FR
url_snomed_fr = f"{FHIR_BASE}/CodeSystem/11000315107-20250621?_format=json"
r_snomed_fr = http_get(url_snomed_fr)
snomed_fr = r_snomed_fr.json()

print("[PREUVE étape 210] Détail CodeSystem SNOMED CT FR :")
print_keys(snomed_fr, "id", "version", "name", "title", "status", "content")

# Étape 220 — Versions détaillées SNOMED CT FR
print("\nVersions détaillées SNOMED CT FR :")
url_versions_fr = f"{WP_BASE}/terminologies/versions-details?terminologyId=terminologie-snomed-ct-fr"
r_versions_fr = http_get(url_versions_fr, HEADERS_API)
versions_fr = r_versions_fr.json()
print(json.dumps(versions_fr, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# Étape 230 — PREUVE : téléchargement SNOMED CT FR zip
snomed_fr_zip_url = (
    f"{WP_BASE}/terminologies/zip"
    "?terminologyId=terminologie-snomed-ct-fr"
    "&version=Juin%202025%20v1.0"
    "&licenceConsent=true"
    "&dataTransferConsent=true"
)
print(f"Téléchargement : {snomed_fr_zip_url}")
r_zip_fr = http_get(snomed_fr_zip_url, HEADERS_API)
with open("snomed-ct-fr.zip", "wb") as f:
    f.write(r_zip_fr.content)
print(f"✓ snomed-ct-fr.zip — {len(r_zip_fr.content) / 1024 / 1024:.1f} MB")

---
## Récapitulatif

In [ ]:
import os

snapshots = {
    "smt-cs.json"    : "CapabilityStatement général (étapes 10-30)",
    "smt-tc.json"    : "CapabilityStatement terminologique (étapes 50-70)",
    "smt-tre256.json": "CodeSystem TRE-R256-TypeMessagerie (étapes 90-150)",
    "snomed-ct.zip"  : "SNOMED CT International (étapes 170-190)",
    "snomed-ct-fr.zip": "SNOMED CT France (étapes 210-230)",
}

print("Fichiers générés :")
print(f"{'Fichier':<25} {'Taille':>10}    Description")
print("-" * 75)
for filename, description in snapshots.items():
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
        print(f"  {filename:<23} {size_str:>10}    {description}")
    else:
        print(f"  {filename:<23} {'MANQUANT':>10}    {description}")

print("\n✓ Cas de test nominal SVCM-Terminology-Consumer terminé.")